In [5]:
import pandas as pd

raw = pd.read_csv("teaching_prices_messy.csv")

In [6]:
df = raw.copy()
df = df.drop_duplicates()
df["price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", "", regex=False), errors="coerce")
df["date"] = pd.to_datetime(df["date"])

df["flag"] = ""
mask = df["item"] == "oil b"
df.loc[mask, "item"] = "Staple A"
df.loc[mask, "flag"] = "relabelled_from_oil_b"
df["item"] = df["item"].str.strip().str.title()

med = df.groupby("item")["price"].transform("median")
df.loc[df["price"].isna(), "flag"] = "missing_price"
df.loc[(df["price"] > 3 * med) | (df["price"] < med / 3), "flag"] = "outlier_check"

df = df.sort_values(["date", "item", "city"]).reset_index(drop=True)
df.to_csv("teaching_prices_clean.csv", index=False)
df[df["flag"] != ""][["date", "item", "city", "price", "flag"]]

,date,item,city,price,flag
3,2026-01-01,Pulse C,City 2,NaN,missing_price
21,2026-01-15,Staple A,City 2,1250.0,outlier_check
30,2026-01-22,Sweetener D,City 1,9999.0,outlier_check
44,2026-02-05,Staple A,City 1,108.5,relabelled_from_oil_b


In [7]:
g = df.groupby("item")["price"]
stats = pd.DataFrame({"mean": g.mean(), "median": g.median()})

def first_to_last_pct(item_df):
    wide = item_df.pivot_table(index="date", columns="city", values="price").dropna()
    avg = wide.mean(axis=1)
    return (avg.iloc[-1] / avg.iloc[0] - 1) * 100

stats["pct_change_first_to_last"] = df.groupby("item").apply(first_to_last_pct)
stats.round(2)

/tmp/ipykernel_174/169048939.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats["pct_change_first_to_last"] = df.groupby("item").apply(first_to_last_pct)


,mean,median,pct_change_first_to_last
item,,,
Oil B,324.83,324.00,15.28
Pulse C,205.91,206.00,2.43
Staple A,158.71,112.50,12.98
Sweetener D,576.81,167.75,19.41
